# Extract Residual Stream Activations

This notebook demonstrates how to capture residual-stream activations during
vLLM inference using the async engine and vllm-lens.

**Requirements:** NVIDIA GPU, Python 3.12+, and a working vLLM install where `import vllm._C` succeeds. On Windows, use CUDA PyTorch from the [PyTorch cu126 index](https://download.pytorch.org/whl/cu126), then `pip install -e .` from this repo so the `vllm.general_plugins` entry point is registered. If the first code cell fails, use the **Local development (Conda + Jupyter)** section in `README.md` at the repo root.

## Imports

In [ ]:
try:
    import vllm._C  # noqa: F401 - native extension; fails on CPU-only / broken installs
except ModuleNotFoundError as e:
    raise RuntimeError(
        "vllm._C is missing: vLLM needs a GPU build with compiled extensions. "
        "On Windows, install CUDA PyTorch from https://download.pytorch.org/whl/cu126, "
        "then reinstall vllm, and install this package with pip install -e . "
        "See the repo README (Local development, Conda + Jupyter)."
    ) from e

import vllm_lens  # noqa: F401 - package must be installed; vLLM loads the plugin at engine startup

from vllm import AsyncEngineArgs, AsyncLLMEngine, SamplingParams

## Configuration

In [ ]:
MODEL = "facebook/opt-125m"
PROMPT = "The future of AI is"
LAYERS = [0, 2, 5]

## Initialize the Async Engine

In [ ]:
engine_args = AsyncEngineArgs(
    model=MODEL,
    gpu_memory_utilization=0.85,
)
engine = AsyncLLMEngine.from_engine_args(engine_args)

## Generate with Activation Capture

Pass `output_residual_stream` via `extra_args` to request residual-stream
activations at the specified layers.

In [ ]:
sampling_params = SamplingParams(
    temperature=0.7,
    max_tokens=64,
    extra_args={"output_residual_stream": LAYERS},
)


async def _run_request():
    final_local = None
    async for output in engine.generate(PROMPT, sampling_params, request_id="req-0"):
        final_local = output
    return final_local


final = await _run_request()

## Inspect Results

In [ ]:
print(f"Prompt: {PROMPT}")
print(f"Generated: {final.outputs[0].text}")

activation_data = getattr(final, "activations", None)
if activation_data is not None:
    residual_stream = activation_data["residual_stream"]
    print(f"\nResidual stream shape: {residual_stream.shape}")
    print(f"Sample activations (layer 0, first token): {residual_stream[0, 0, :5]}")
else:
    print("Activations: NOT FOUND (ERROR)")